In [12]:
from tensorflow.keras.datasets import mnist
import tracemalloc
import numpy as np

(_, _), (images, labels) = mnist.load_data()

image_i = 40
image = images[image_i]
label = labels[image_i]

In [13]:
def check_center(image):
    center_size = 8
    start = (image.shape[0] - center_size) // 2
    end = start + center_size
    center_square = image[start:end, start:end]
    score = center_square.mean() / 255.0
    return score

def vertical_symmetry(image):
    flipped = np.fliplr(image)
    diff = np.abs(image - flipped)
    score = 1 - (diff.mean() / 255.0)
    return score

def horizontal_symmetry(image):
    flipped = np.flipud(image)
    diff = np.abs(image - flipped)
    score = 1 - (diff.mean() / 255.0)
    return score

def top_bottom_balance(image):
    mid = image.shape[0] // 2
    top = image[:mid, :].mean()
    bottom = image[mid:, :].mean()
    return (top - bottom) / 255.0

def pixel_density(image):
    return image.mean() / 255.0

def left_right_balance(image):
    mid = image.shape[1] // 2
    left = image[:, :mid].mean()
    right = image[:, mid:].mean()
    return (left - right) / 255.0

def active_pixels(image):
    return np.sum(image > 50) / (28 * 28)

In [14]:
sorted_features = {'center': {0: np.float64(0.19696572724142808), 1: np.float64(0.38121493665695205), 2: np.float64(0.3867747434656973), 3: np.float64(0.4315763609717233), 4: np.float64(0.4902966118405597), 5: np.float64(0.39078678233883374), 6: np.float64(0.45033432813814883), 7: np.float64(0.327086199004742), 8: np.float64(0.5456597728392341), 9: np.float64(0.4749486958592479)}, 'vertical_sim': {0: np.float64(0.8415833664210184), 1: np.float64(0.917646655220974), 2: np.float64(0.8584043217824232), 3: np.float64(0.8611562191342018), 4: np.float64(0.8798918129388028), 5: np.float64(0.8643233423234622), 6: np.float64(0.8699486012722093), 7: np.float64(0.8835874397644162), 8: np.float64(0.8563493351536473), 9: np.float64(0.880698408124389)}, 'horizontal_sim': {0: np.float64(0.8439645181050605), 1: np.float64(0.9226739286637177), 2: np.float64(0.8451648400212707), 3: np.float64(0.8560661256836741), 4: np.float64(0.8744227852044628), 5: np.float64(0.8539559413508974), 6: np.float64(0.8621073573058848), 7: np.float64(0.8739718457215294), 8: np.float64(0.847904870374055), 9: np.float64(0.8747646579223368)}, 'top-bottom-bal': {0: np.float64(-0.014093023669031987), 1: np.float64(-0.007918327520863013), 2: np.float64(-0.054235601592097275), 3: np.float64(-0.005900863036455822), 4: np.float64(-0.03014881577417009), 5: np.float64(-0.004269416676465839), 6: np.float64(-0.056320674603061935), 7: np.float64(0.020665990258832978), 8: np.float64(-0.005579676751917639), 9: np.float64(-0.002416248059149701)}, 'left-right-bal': {0: np.float64(-0.014928191435277419), 1: np.float64(-0.03253005889392241), 2: np.float64(-0.03066231192040422), 3: np.float64(-0.0436756891632856), 4: np.float64(-0.027995177872586822), 5: np.float64(-0.013628596614812633), 6: np.float64(-0.010928763191657165), 7: np.float64(-0.039679581329738596), 8: np.float64(-0.01847369096331064), 9: np.float64(-0.0390216954154329)}, 'density': {0: np.float64(0.1733993251192091), 1: np.float64(0.07599864255996085), 2: np.float64(0.1489751288229288), 3: np.float64(0.14153014329202557), 4: np.float64(0.1213655909128458), 5: np.float64(0.12874939405756802), 6: np.float64(0.13730177522174766), 7: np.float64(0.1145276977510873), 8: np.float64(0.15015598189369683), 9: np.float64(0.12258994285224575)}, 'active': {0: np.float64(0.21112633386969404), 1: np.float64(0.09394581030276278), 2: np.float64(0.1835739205047576), 3: np.float64(0.17572228287824657), 4: np.float64(0.15187479476556154), 5: np.float64(0.16246037706726282), 6: np.float64(0.16948487319902594), 7: np.float64(0.14204582471456245), 8: np.float64(0.1864847802050229), 9: np.float64(0.15342893334842783)}}

sort_method = {
    "center": check_center,
    "vertical_sim": vertical_symmetry,
    "horizontal_sim": horizontal_symmetry,
    "top-bottom-bal": top_bottom_balance,
    "left-right-bal": left_right_balance,
    "density": pixel_density,
    "active": active_pixels,
}

In [18]:
features = [
    check_center(image),
    vertical_symmetry(image),
    horizontal_symmetry(image),
    top_bottom_balance(image),
    left_right_balance(image),
    pixel_density(image),
    active_pixels(image)
]

def classify(image):
    num_scores = {0: 0, 1: 0, 2:0, 3:0, 4:0, 5:0, 6:0, 7:0, 8:0, 9:0}

    for method in sort_method:
        result = sort_method[method](image)
        closest_key = min(sorted_features[method], key=lambda k: abs(sorted_features[method][k] - result))
        num_scores[closest_key] += 1


    highest_freq = max(num_scores.values())
    highest_freq_index = list(num_scores.values()).index(highest_freq)
    key = list(num_scores.keys())[highest_freq_index]
    return key


tracemalloc.start()
tracemalloc.clear_traces()

print(f"classified: {classify(images[40])}\nactual: {label}")

curr, peak = tracemalloc.get_traced_memory()

print(f"curr: {curr/1000_000} mb | max {peak/1000_000} mb | differance {(peak-curr)/1000_000}")

classified: 1
actual: 1
curr: 0.004001 mb | max 0.024299 mb | differance 0.020298


# Analyse

Met eem maximaal gebruikt van 0.025 mb valt deze aanpak ruimschoots binnen het maximale geheugen gebruik van 1 mb.